# 03 - Extraction des caracteristiques


Calculer des signatures couleur HSV et texture LBP. Les parametres sont a tester.

**Bibliothèques utilisées**

- Pillow ouvre le fichier et prépare ses trois représentations. NumPy convertit les images en tableaux et compte les valeurs dans les histogrammes. scikit-image applique le descripteur LBP et Matplotlib trace les figures.
- La cellule d’installation s’exécute avant les imports. Si tu as déjà installé les paquets dans une autre session, vérifie que le noyau actif utilise le même environnement.

In [ ]:
%pip install pillow numpy matplotlib scikit-image

**Des images Pillow aux tableaux de pixels**

- Le fichier est ouvert une fois, converti en RGB et redimensionné à 256 × 256 avec Lanczos. Le test sur le chemin produit un message explicite quand le notebook est lancé depuis le mauvais répertoire.
- Après conversion et passage par `np.asarray`, le tableau `hsv` a la forme **(256, 256, 3)**. Les deux premiers axes désignent la ligne et la colonne du pixel. Le troisième contient H, S et V, codés par Pillow sur des entiers de 0 à 255.
- Le tableau `gris` a la forme **(256, 256)**, avec une intensité par pixel. L’image RGB affichée indique de quel fichier proviennent les deux signatures.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.feature import local_binary_pattern

chemin = Path("data/101_ObjectCategories/airplanes/image_0001.jpg")
if not chemin.is_file():
    raise FileNotFoundError(f"Image introuvable : {chemin.resolve()}. Ouvre le notebook depuis la racine du projet.")
with Image.open(chemin) as image:
    rgb = image.convert("RGB").resize((256, 256), Image.Resampling.LANCZOS)
hsv = np.asarray(rgb.convert("HSV"))
gris = np.asarray(rgb.convert("L"))
plt.imshow(rgb)
plt.axis("off")
plt.title("Image redimensionnee")
plt.show()

## Signature couleur HSV


Histogramme 3D : 8 intervalles par canal, soit 512 valeurs. La somme est normalisee a 1.

**Histogramme conjoint de couleur HSV**

- `reshape(-1, 3)` réorganise les pixels en **65 536 lignes** de trois valeurs. `np.histogramdd` place chaque triplet dans une case de l’espace HSV. Chaque canal possède huit intervalles entre 0 et 256, soit **8 × 8 × 8 = 512 cases**. Deux pixels de même teinte peuvent tomber dans des cases différentes si leur saturation ou leur valeur change.
- Après division par le nombre total de pixels, chaque case représente une proportion. `ravel()` transforme la grille 8 × 8 × 8 en vecteur, avec un ordre fixe des cases. L’assertion vérifie la longueur et la somme proche de 1.
- Le graphique ne montre que la projection sur H. `sum(axis=(1, 2))` additionne, pour chaque intervalle de teinte, toutes les valeurs possibles de S et V. Le vecteur comparé plus tard conserve les 512 cases.

In [ ]:
hist_hsv, _ = np.histogramdd(
    hsv.reshape(-1, 3), bins=(8, 8, 8), range=((0, 256), (0, 256), (0, 256))
)
hist_hsv = hist_hsv / hist_hsv.sum()
signature_couleur = hist_hsv.ravel()
assert signature_couleur.shape == (512,) and np.isclose(signature_couleur.sum(), 1)
print("Signature couleur :", signature_couleur.size, "valeurs, somme :", signature_couleur.sum())
plt.bar(range(8), hist_hsv.sum(axis=(1, 2)))
plt.xticks(range(8))
plt.xlabel("Intervalle de teinte H")
plt.ylabel("Proportion de pixels")
plt.title("Projection de la signature HSV sur la teinte")
plt.show()

## Signature de texture LBP


LBP compare chaque pixel a ses voisins. La methode uniforme avec 8 voisins et un rayon de 1 donne 10 valeurs normalisees.

**Histogramme de texture LBP uniforme**

- `local_binary_pattern(gris, 8, 1, method="uniform")` compare l’intensité de huit voisins à celle du pixel central. Le rayon de 1 fixe la distance des voisins. Les comparaisons donnent une suite de bits qui décrit le motif local.
- Dans le mode uniforme, un motif avec au plus deux transitions entre 0 et 1 autour du cercle est codé par le nombre de bits à 1, de 0 à 8. Les autres motifs partagent le code 9. `np.arange(voisins + 3)` fournit onze bords pour les dix cases.
- La carte de gauche montre le code attribué à chaque position. Le diagramme de droite compte les proportions par code. Le vecteur de dix valeurs mesure la fréquence des motifs dans l’image entière et perd leur position exacte.

In [ ]:
voisins, rayon = 8, 1
lbp = local_binary_pattern(gris, voisins, rayon, method="uniform")
signature_texture, _ = np.histogram(lbp, bins=np.arange(voisins + 3))
signature_texture = signature_texture / signature_texture.sum()
assert signature_texture.shape == (10,) and np.isclose(signature_texture.sum(), 1)
print("Signature texture :", signature_texture.size, "valeurs, somme :", signature_texture.sum())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(lbp, cmap="gray")
axes[0].set_title("Codes LBP")
axes[0].axis("off")
axes[1].bar(range(len(signature_texture)), signature_texture)
axes[1].set_xlabel("Code LBP uniforme")
axes[1].set_ylabel("Proportion de pixels")
axes[1].set_title("Histogramme de texture")
plt.tight_layout()
plt.show()

Ces signatures sont globales : elles ne conservent pas precisement la position des couleurs ou des textures. La prochaine etape sera de comparer plusieurs images.